In [1]:
import pandas as pd
import numpy as np

from statsmodels.tsa.stattools import pacf

from sklearn.feature_selection import mutual_info_regression
from sklearn.ensemble import RandomForestRegressor


from sklearn.preprocessing import MinMaxScaler, RobustScaler, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from xgboost import XGBRegressor

from sklearn.tree import DecisionTreeRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
import time

# Read file

In [2]:



file_path = './dataset/observations_part_001.parquet'
df_raw = pd.read_parquet(file_path)

print(f"Row Total: {df_raw.shape[0]:,}")
print(f"Column Total: {df_raw.shape[1]}")

Row Total: 480,000
Column Total: 59


# Data Info


In [3]:
df_raw.info()



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 480000 entries, 0 to 479999
Data columns (total 59 columns):
 #   Column                        Non-Null Count   Dtype  
---  ------                        --------------   -----  
 0   activity_multiplier           480000 non-null  float32
 1   animal_residue_pct            480000 non-null  int32  
 2   battery_level_percent         477785 non-null  float32
 3   battery_maintenance_event     480000 non-null  int32  
 4   bin_index                     480000 non-null  int32  
 5   collection_delay_minutes      480000 non-null  float32
 6   collection_event              480000 non-null  int32  
 7   collection_threshold_percent  480000 non-null  float32
 8   college_calendar_closed       480000 non-null  int32  
 9   date_index                    480000 non-null  int32  
 10  day_of_week                   480000 non-null  int32  
 11  dominant_waste_code           480000 non-null  int32  
 12  event_code                    480000 non-nul

In [4]:
df_raw.head()

,activity_multiplier,animal_residue_pct,battery_level_percent,battery_maintenance_event,bin_index,collection_delay_minutes,collection_event,collection_threshold_percent,college_calendar_closed,date_index,...,sensor_error_code,temperature_c,textile_pct,time_slot_code,timestamp_unix,true_fill_level_percent,waste_bulk_density_kg_m3,waste_entering_bin_kg,waste_volume_liter,weather_code
0,0.6984,4,96.482681,0,0,0.0,0,80.0,0,4,...,0,13.053044,6,0,1736013600,49.544697,226.006271,0.629307,2.784469,0
1,0.5626,0,92.744682,0,1,0.0,0,80.0,0,4,...,0,14.072392,1,0,1736013600,49.981346,251.556946,0.622490,2.474549,0
2,0.8924,1,96.756447,0,2,0.0,0,80.0,0,4,...,0,13.722422,4,0,1736013600,32.794777,232.091263,1.656849,7.138784,0
3,0.2134,3,95.200546,0,3,0.0,0,75.0,0,4,...,0,13.480151,4,0,1736013600,24.558718,199.116287,0.294287,1.477966,0
4,0.8924,0,92.272179,0,4,0.0,0,75.0,0,4,...,0,13.799287,3,0,1736013600,67.464378,222.365173,2.896791,13.027180,0


# Cleaning Data
Ambil 50 tempat sampah dan urutkan


In [5]:
df_50 = df_raw[df_raw['bin_index'] < 50].sort_values(['bin_index', 'timestamp_unix']).reset_index(drop=True)

## Anomaly Handing
Hapus nilai jika sensor error/packet loss, lalu lakukan interpolasi (pengisian nilai tengah)

In [6]:
anomaly_mask = (df_50['is_sensor_anomaly'] == 1) | (df_50['is_packet_loss'] == 1)
print(f"-> Ditemukan {anomaly_mask.sum()} baris anomali/error hardware. Melakukan Interpolasi...")
df_50.loc[anomaly_mask, 'fill_level_percent'] = np.nan
df_50['fill_level_percent'] = df_50.groupby('bin_index')['fill_level_percent'].transform(
    lambda x: x.interpolate(method='linear').bfill().ffill()
)

-> Ditemukan 28 baris anomali/error hardware. Melakukan Interpolasi...


### Statistical Justification
Uji PACF untuk menjustifikasi kenapa pakai riwayat 3 Jam


In [7]:
pacf_vals = pacf(df_50[df_50['bin_index'] == 0]['fill_level_percent'].dropna(), nlags=5)
print(f"Uji PACF (Partial Autocorrelation) pada Bin 0:")
for i, val in enumerate(pacf_vals):
    print(f"  Lag {i} jam lalu: {val:.3f} (Jika > 0.1, artinya signifikan secara statistik)")

Uji PACF (Partial Autocorrelation) pada Bin 0:
  Lag 0 jam lalu: 1.000 (Jika > 0.1, artinya signifikan secara statistik)
  Lag 1 jam lalu: 0.927 (Jika > 0.1, artinya signifikan secara statistik)
  Lag 2 jam lalu: -0.021 (Jika > 0.1, artinya signifikan secara statistik)
  Lag 3 jam lalu: -0.040 (Jika > 0.1, artinya signifikan secara statistik)
  Lag 4 jam lalu: -0.053 (Jika > 0.1, artinya signifikan secara statistik)
  Lag 5 jam lalu: -0.020 (Jika > 0.1, artinya signifikan secara statistik)


In [8]:
df_proc = df_50.copy()

#### A. Cyclical Time Encoding (Mencegah AI bingung perpindahan Jam 23 ke Jam 0)

In [9]:
df_proc['hour_sin'] = np.sin(2 * np.pi * df_proc['hour'] / 24.0)
df_proc['hour_cos'] = np.cos(2 * np.pi * df_proc['hour'] / 24.0)
df_proc['day_sin'] = np.sin(2 * np.pi * df_proc['day_of_week'] / 7.0)
df_proc['day_cos'] = np.cos(2 * np.pi * df_proc['day_of_week'] / 7.0)

#### B. Lag Features


In [10]:
df_proc['lag_1h'] = df_proc.groupby('bin_index')['fill_level_percent'].shift(1)
df_proc['lag_2h'] = df_proc.groupby('bin_index')['fill_level_percent'].shift(2)
df_proc['lag_3h'] = df_proc.groupby('bin_index')['fill_level_percent'].shift(3)

#### C. Ekstraksi Velocity (Laju percepatan sampah masuk)


In [11]:
df_proc['velocity_1h'] = df_proc['fill_level_percent'] - df_proc['lag_1h']
df_proc['velocity_2h'] = df_proc['lag_1h'] - df_proc['lag_2h']

#### D. Pembuatan Target Output (Kapasitas 1 Jam ke depan)


In [12]:
df_proc['target_next_hour'] = df_proc.groupby('bin_index')['fill_level_percent'].shift(-1)

##### Buang NaN


In [13]:
df_proc = df_proc.dropna().reset_index(drop=True)

### Feature Selection

In [14]:
# Set 1: FE Minimal (Hanya 1 Jam memori & Waktu Mentah)
fe_set_1 = [
    'fill_level_percent', 'lag_1h', 'hour', 'day_of_week'
]

# Set 2: FE Kompleks (Multi-Lag, Velocity, & Cyclical Time)
fe_set_2 = [
    'fill_level_percent', 'lag_1h', 'lag_2h', 'lag_3h',
    'velocity_1h', 'velocity_2h', 
    'hour_sin', 'hour_cos', 'day_sin', 'day_cos'
]

Mengambil sampel 5000 baris agar komputasi MI cepat


In [15]:
n_sample = min(5000, len(df_proc))
sampel_mi = df_proc.sample(n_sample, random_state=42)

mi_scores = mutual_info_regression(sampel_mi[fe_set_2], sampel_mi['target_next_hour'], random_state=42)
mi_df = pd.DataFrame({'Fitur': fe_set_2, 'Skor_MI': mi_scores}).sort_values('Skor_MI', ascending=False)
print("Information Gain tiap fitur terhadap Target (Nilai lebih tinggi = Lebih penting):")
print(mi_df.to_string(index=False))

Information Gain tiap fitur terhadap Target (Nilai lebih tinggi = Lebih penting):
             Fitur  Skor_MI
fill_level_percent 2.018102
            lag_1h 1.794074
            lag_2h 1.591467
            lag_3h 1.434640
       velocity_1h 0.033735
       velocity_2h 0.028473
           day_cos 0.000136
          hour_sin 0.000000
          hour_cos 0.000000
           day_sin 0.000000


## Train-Test Split


In [16]:
train_mask = df_proc['bin_index'] < 40  # 40 Bin untuk Belajar
test_mask = df_proc['bin_index'] >= 40  # 10 Bin untuk Ujian

y_train = df_proc.loc[train_mask, 'target_next_hour']
y_test = df_proc.loc[test_mask, 'target_next_hour']

## Experimental A (Using Raw Data)

In [17]:
# Murni data mentah dari sensor tanpa memori
basic_cols = ['fill_level_percent', 'hour', 'day_of_week']

rf_baseline = RandomForestRegressor(n_estimators=50, random_state=42, n_jobs=-1)
rf_baseline.fit(df_proc.loc[train_mask, basic_cols], y_train)

pred_basic = rf_baseline.predict(df_proc.loc[test_mask, basic_cols])
mae_basic = mean_absolute_error(y_test, pred_basic)

## Experimental B (Using FE Set 1 vs FE Set 2)

In [18]:
# Test FE Set 1
rf_fe1 = RandomForestRegressor(n_estimators=50, random_state=42, n_jobs=-1)
rf_fe1.fit(df_proc.loc[train_mask, fe_set_1], y_train)
mae_fe1 = mean_absolute_error(y_test, rf_fe1.predict(df_proc.loc[test_mask, fe_set_1]))

# Test FE Set 2
rf_fe2 = RandomForestRegressor(n_estimators=50, random_state=42, n_jobs=-1)
rf_fe2.fit(df_proc.loc[train_mask, fe_set_2], y_train)
mae_fe2 = mean_absolute_error(y_test, rf_fe2.predict(df_proc.loc[test_mask, fe_set_2]))

print("=== EVALUASI REKAYASA FITUR ===")
print(f"1. Tanpa Feature Engineering (Mentah) : MAE {mae_basic:.3f}%")
print(f"2. Feature Engineering Set 1 (Minimal): MAE {mae_fe1:.3f}%")
print(f"3. Feature Engineering Set 2 (Kompleks): MAE {mae_fe2:.3f}%")

=== EVALUASI REKAYASA FITUR ===
1. Tanpa Feature Engineering (Mentah) : MAE 3.674%
2. Feature Engineering Set 1 (Minimal): MAE 3.479%
3. Feature Engineering Set 2 (Kompleks): MAE 3.146%


## Experimental C (Comprehensive Model & Scaler Comparison)
Menguji Linear Regression, Random Forest, dan XGBoost terhadap berbagai jenis Feature Scaling.


In [19]:
# 1. Dictionary of Scalers
scalers_dict = {
    'Tanpa Scaler': None,
    'StandardScaler': StandardScaler(),
    'MinMaxScaler': MinMaxScaler(),
    'RobustScaler': RobustScaler()
}

# 2. Dictionary of Models
models_dict = {
    'Linear Regression': LinearRegression(),
    'Decision Tree': DecisionTreeRegressor(random_state=42),
    'Random Forest': RandomForestRegressor(random_state=42, n_jobs=-1),
    'XGBoost': XGBRegressor(random_state=42, n_jobs=-1),
    'LightGBM': LGBMRegressor(random_state=42, n_jobs=-1, verbose=-1),
    'CatBoost': CatBoostRegressor(random_state=42, verbose=0)
}

grid_results = []
print("Mulai melatih model secara efisien... Mohon tunggu.")

# Ekstrak data mentah untuk model Tree
X_tr_raw = df_proc.loc[train_mask, fe_set_2]
X_te_raw = df_proc.loc[test_mask, fe_set_2]

# Latih semua model Tree HANYA SATU KALI (Tanpa Scaler) untuk menghemat waktu
tree_results = {}
for model_name, model_obj in models_dict.items():
    if model_name != 'Linear Regression':
        start_time = time.time()
        model_obj.fit(X_tr_raw, y_train)
        pred = model_obj.predict(X_te_raw)
        duration = time.time() - start_time
        mae_score = mean_absolute_error(y_test, pred)
        tree_results[model_name] = f"{mae_score:.3f}% ({duration:.2f}s)"

# Bangun baris demi baris untuk tabel
for scaler_name, scaler_obj in scalers_dict.items():
    row_result = {'Scaler': scaler_name}
    
    # Proses Linear Regression (Dijalankan untuk semua scaler)
    if scaler_obj is None:
        X_tr, X_te = X_tr_raw, X_te_raw
    else:
        X_tr = scaler_obj.fit_transform(X_tr_raw)
        X_te = scaler_obj.transform(X_te_raw)
        
    start_time = time.time()
    lr_model = LinearRegression()
    lr_model.fit(X_tr, y_train)
    pred = lr_model.predict(X_te)
    duration = time.time() - start_time
    mae_score = mean_absolute_error(y_test, pred)
    row_result['Linear Regression'] = f"{mae_score:.3f}% ({duration:.2f}s)"
    
    # Isi kolom model Tree
    for model_name in models_dict.keys():
        if model_name != 'Linear Regression':
            if scaler_name == 'Tanpa Scaler':
                row_result[model_name] = tree_results[model_name]
            else:
                row_result[model_name] = "--- Dilewati ---"
                
    grid_results.append(row_result)

Mulai melatih model secara efisien... Mohon tunggu.


# Evaluation Result

In [20]:
# Tampilkan Tabel Grid Hasil Scaler & Model Lanjutan
print("\n=== ARENA KOMPARASI: 6 MODEL vs 4 SCALER (OPTIMIZED) ===")
print("* Format isi tabel: [Nilai Meleset MAE] (Waktu Training & Prediksi)")
print("* Catatan: Pengujian Scaler pada model Tree dilewati untuk menghindari computational noise.\n")

df_grid_results = pd.DataFrame(grid_results)
df_grid_results.set_index('Scaler', inplace=True)
print(df_grid_results.to_string())


=== ARENA KOMPARASI: 6 MODEL vs 4 SCALER (OPTIMIZED) ===
* Format isi tabel: [Nilai Meleset MAE] (Waktu Training & Prediksi)
* Catatan: Pengujian Scaler pada model Tree dilewati untuk menghindari computational noise.

               Linear Regression     Decision Tree     Random Forest           XGBoost          LightGBM          CatBoost
Scaler                                                                                                                    
Tanpa Scaler      3.198% (0.01s)    4.342% (0.06s)    3.119% (0.61s)    3.202% (0.24s)    3.070% (0.10s)    3.072% (2.46s)
StandardScaler    3.198% (0.00s)  --- Dilewati ---  --- Dilewati ---  --- Dilewati ---  --- Dilewati ---  --- Dilewati ---
MinMaxScaler      3.198% (0.00s)  --- Dilewati ---  --- Dilewati ---  --- Dilewati ---  --- Dilewati ---  --- Dilewati ---
RobustScaler      3.198% (0.00s)  --- Dilewati ---  --- Dilewati ---  --- Dilewati ---  --- Dilewati ---  --- Dilewati ---


# Save Dataset

In [21]:
df_proc.head()

,activity_multiplier,animal_residue_pct,battery_level_percent,battery_maintenance_event,bin_index,collection_delay_minutes,collection_event,collection_threshold_percent,college_calendar_closed,date_index,...,hour_sin,hour_cos,day_sin,day_cos,lag_1h,lag_2h,lag_3h,velocity_1h,velocity_2h,target_next_hour
0,0.6984,1,96.396736,0,0,0.0,0,80.0,0,4,...,0.707107,7.071068e-01,-0.781831,0.62349,52.753166,50.935368,49.665939,-1.902481,1.817799,53.736298
1,0.6984,0,96.368088,0,0,0.0,0,80.0,0,4,...,0.866025,5.000000e-01,-0.781831,0.62349,50.850685,52.753166,50.935368,2.885612,-1.902481,54.325520
2,0.6984,1,96.339439,0,0,0.0,0,80.0,0,4,...,0.965926,2.588190e-01,-0.781831,0.62349,53.736298,50.850685,52.753166,0.589222,2.885612,53.611057
3,0.6984,2,96.310791,0,0,0.0,0,80.0,0,4,...,1.000000,6.123234e-17,-0.781831,0.62349,54.325520,53.736298,50.850685,-0.714462,0.589222,55.610485
4,0.6984,2,96.282143,0,0,0.0,0,80.0,0,4,...,0.965926,-2.588190e-01,-0.781831,0.62349,53.611057,54.325520,53.736298,1.999428,-0.714462,56.233257


In [23]:
trainable_feat = ['bin_index', 'timestamp_unix'] + fe_set_2 + ['target_next_hour']

df_clean = df_proc[trainable_feat].copy()

df_clean.to_csv('dataset_trainable_rev1.csv', index=False)

print(f"Sudah memasuki ke dalam bentuk csv {df_clean.shape[1]} kolom, siap training")


Sudah memasuki ke dalam bentuk csv 13 kolom, siap training


In [24]:
df_clean.head()

,bin_index,timestamp_unix,fill_level_percent,lag_1h,lag_2h,lag_3h,velocity_1h,velocity_2h,hour_sin,hour_cos,day_sin,day_cos,target_next_hour
0,0,1736024400,50.850685,52.753166,50.935368,49.665939,-1.902481,1.817799,0.707107,7.071068e-01,-0.781831,0.62349,53.736298
1,0,1736028000,53.736298,50.850685,52.753166,50.935368,2.885612,-1.902481,0.866025,5.000000e-01,-0.781831,0.62349,54.325520
2,0,1736031600,54.325520,53.736298,50.850685,52.753166,0.589222,2.885612,0.965926,2.588190e-01,-0.781831,0.62349,53.611057
3,0,1736035200,53.611057,54.325520,53.736298,50.850685,-0.714462,0.589222,1.000000,6.123234e-17,-0.781831,0.62349,55.610485
4,0,1736038800,55.610485,53.611057,54.325520,53.736298,1.999428,-0.714462,0.965926,-2.588190e-01,-0.781831,0.62349,56.233257
